# E-commerce Medallion Pipeline — Databricks Serverless

Run **Bronze → Silver → Gold** from your repo clone.

**Serverless constraints (this notebook handles all of them):**
- Do **not** use `dbfs:/FileStore/...` (public DBFS is disabled)
- Do **not** use `/tmp` or `file:/tmp` (Spark cannot read them)
- CSVs must live under **`{REPO_ROOT}/data`** (workspace path)
- Pull latest `main` from GitHub before running

**Steps:** Run every cell top-to-bottom. Use **Run all** after Repos → Pull.

In [ ]:
# =============================================================================
# CELL 1 — Setup (run first)
# =============================================================================
import os
import sys
from pathlib import Path

# Legacy widget kept for old cells / job parameters (value is optional)
dbutils.widgets.text("source_base_path", "", "Leave empty = {REPO_ROOT}/data")

notebook_path = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook()
    .getContext()
    .notebookPath()
    .get()
)

# Works for /Workspace/Repos/... and /Workspace/Users/.../repo clones
repo_rel = os.path.dirname(os.path.dirname(notebook_path))
REPO_ROOT = repo_rel if repo_rel.startswith("/Workspace") else f"/Workspace{repo_rel}"
SRC_ROOT = os.path.join(REPO_ROOT, "src")
DATA_DIR = Path(REPO_ROOT) / "data"

if not os.path.isdir(SRC_ROOT):
    raise FileNotFoundError(
        f"src not found at {SRC_ROOT}. Open this notebook from the Git repo clone."
    )

if SRC_ROOT not in sys.path:
    sys.path.insert(0, SRC_ROOT)

os.environ["PIPELINE_REPO_ROOT"] = REPO_ROOT

# Python 3.12 on serverless: register config module before dataclass use
import config.pipeline_config  # noqa: F401
import config.databricks_runtime  # noqa: F401

DATA_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_BASE_PATH_WIDGET = dbutils.widgets.get("source_base_path").strip()
if SOURCE_BASE_PATH_WIDGET and "/FileStore" not in SOURCE_BASE_PATH_WIDGET and not SOURCE_BASE_PATH_WIDGET.startswith(("/tmp", "file:/tmp")):
    SOURCE_BASE_PATH = SOURCE_BASE_PATH_WIDGET
else:
    from config.databricks_runtime import to_spark_readable_path

    SOURCE_BASE_PATH = to_spark_readable_path(str(DATA_DIR.resolve()))

print(f"notebook_path={notebook_path}")
print(f"REPO_ROOT={REPO_ROOT}")
print(f"SRC_ROOT={SRC_ROOT}")
print(f"DATA_DIR={DATA_DIR}")
print(f"SOURCE_BASE_PATH={SOURCE_BASE_PATH}")

In [ ]:
# =============================================================================
# CELL 2 — Configuration widgets
# =============================================================================
dbutils.widgets.text("schema_name", "ecommerce", "Hive schema")
dbutils.widgets.dropdown("generate_sample_data", "true", ["true", "false"], "Generate seed=42 CSVs")
dbutils.widgets.text("catalog", "", "Unity Catalog (optional)")
dbutils.widgets.text("run_id", "", "Run id (optional)")

SCHEMA_NAME = dbutils.widgets.get("schema_name").strip() or "ecommerce"
GENERATE_SAMPLE_DATA = dbutils.widgets.get("generate_sample_data") == "true"
CATALOG = dbutils.widgets.get("catalog").strip() or None
RUN_ID = dbutils.widgets.get("run_id").strip() or None

print(f"schema={SCHEMA_NAME}")
print(f"generate_sample_data={GENERATE_SAMPLE_DATA}")
print(f"catalog={CATALOG or '(default)'}")

In [ ]:
# =============================================================================
# CELL 3 — Generate sample CSVs into repo data/ (serverless-safe)
# =============================================================================
if GENERATE_SAMPLE_DATA:
    from data_generation.generate_sample_data import write_sample_datasets, DEFAULT_SEED

    print(f"Generating seed={DEFAULT_SEED} CSVs to {DATA_DIR} ...")
    write_sample_datasets(DATA_DIR, seed=DEFAULT_SEED)

    for name in ("customers.csv", "products.csv", "orders.csv"):
        path = DATA_DIR / name
        if not path.is_file():
            raise FileNotFoundError(f"Expected generated file missing: {path}")
        print(f"OK {path}")
else:
    missing = [n for n in ("customers.csv", "products.csv", "orders.csv") if not (DATA_DIR / n).is_file()]
    if missing:
        raise FileNotFoundError(
            f"Missing CSVs in {DATA_DIR}: {missing}. Set generate_sample_data=true or copy files there."
        )
    print(f"Using existing CSVs in {DATA_DIR}")

In [ ]:
# =============================================================================
# CELL 4 — Run full pipeline (Bronze → Silver → Gold)
# =============================================================================
import logging

from config.pipeline_config import load_config
from run_pipeline import configure_logging, run_pipeline

configure_logging("INFO")

config = load_config(
    source_base_path=SOURCE_BASE_PATH,
    catalog=CATALOG,
    schema_name=SCHEMA_NAME,
    run_id=RUN_ID,
)

print(f"Ingest path (before prepare): {config.source_base_path}")

summary = run_pipeline(
    config,
    spark=spark,
    repo_root=REPO_ROOT,
    generate_sample_data_flag=False,
    validate_sample_data=True,
    strict_sample_row_counts=False,
)

print("\n=== Pipeline summary ===")
print(f"run_id: {summary.run_id}")
print(f"batch_id: {summary.batch_id}")
print(f"source_base_path: {summary.source_base_path}")
print(f"elapsed_seconds: {summary.elapsed_seconds:.1f}")
print(f"steps: {summary.steps_completed}")
print(f"bronze_row_counts: {summary.bronze_row_counts}")
print(f"silver_row_counts: {summary.silver_row_counts}")
print(f"gold_row_counts: {summary.gold_row_counts}")

In [ ]:
# =============================================================================
# CELL 5 — Quick SQL validation
# =============================================================================
spark.sql(f"USE {SCHEMA_NAME}")

display(spark.sql("""
    SELECT 'bronze_customers' AS table_name, COUNT(*) AS row_count FROM bronze_customers
    UNION ALL SELECT 'bronze_products', COUNT(*) FROM bronze_products
    UNION ALL SELECT 'bronze_orders', COUNT(*) FROM bronze_orders
    UNION ALL SELECT 'silver_orders', COUNT(*) FROM silver_orders
    UNION ALL SELECT 'gold_sales_by_product', COUNT(*) FROM gold_sales_by_product
    UNION ALL SELECT 'gold_customer_segmentation', COUNT(*) FROM gold_customer_segmentation
"""))

display(spark.sql("""
    SELECT check_name, failed_records, pass_percentage
    FROM silver_dq_report
    WHERE failed_records > 0
    ORDER BY failed_records DESC
"""))